# 01 — Pembersihan Data (Data Cleaning)

**Proyek:** Siapa yang Paling Sulit Dapat Kerja di Indonesia?  
Analisis Tingkat Pengangguran Terbuka (TPT) Berdasarkan Pendidikan, Usia, Gender, dan Wilayah

---

Notebook ini mengubah 12 file CSV mentah BPS menjadi 10 tabel analisis yang rapi di `data/processed/`.

**Prinsip yang dipegang:**

1. File di `data/raw/` **tidak pernah diubah**. Semua proses baca saja.
2. **Tidak ada imputasi.** Sel yang di BPS berisi `-` dibiarkan kosong dan alasannya dicatat.
3. Semua logika pembersihan ada di `src/cleaning.py` (bukan di notebook), supaya bisa dipakai ulang dan diuji.
4. Setiap baris hasil menyimpan `sumber_file`, jadi angka di dashboard/README bisa ditelusuri balik ke file mentahnya.

> Notebook ini hanya untuk **narasi + verifikasi**. Kalau ingin membangun ulang seluruh data:
> `python -m src.cleaning`

## 1. Kenapa CSV BPS tidak bisa langsung dipakai?

Empat masalah utama yang harus dibereskan lebih dulu:

| # | Masalah | Contoh | Solusi |
|---|---------|--------|--------|
| 1 | Format angka gaya Inggris | `"1,437,402.00"`, `"-"` | hapus koma, `-` jadi `NaN` |
| 2 | Layout bertumpuk | satu file berisi 17 tahun x 2 periode, tiap blok 10 baris | deteksi blok, ubah *wide* ke *long* |
| 3 | Label tidak konsisten | `SLTP` vs `SMP`, `SLTA Umum/SMU` vs `SMA umum`, `D I Y` vs `DI Yogyakarta` | kamus standardisasi |
| 4 | Kategori berubah antar tahun | 2025 menggabungkan SD/tidak tamat SD jadi satu | kolom tambahan `kelompok_pendidikan` 6 level |

Masalah ke-4 yang paling berbahaya: kalau dipaksa dibandingkan apa adanya, grafik tren 2024 vs 2025 akan menyesatkan.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Notebook dijalankan dari folder notebooks/, src/ ada di folder induk
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.cleaning import (
    RAW, PROC,
    read_raw, find_blocks, parse_kegiatan, CATATAN,
    build_nasional_pendidikan, build_provinsi, build_umur,
    build_status_pekerjaan, build_snapshot_2025,
    build_informal_2023, build_setengah_pengangguran_2025,
    parse_triwulanan, parse_tpak_provinsi,
    validate, cek_konsistensi, cek_jumlah_bagian,
)

print("pandas", pd.__version__)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("Folder raw      :", RAW)
print("Folder processed:", PROC)

pandas 3.0.6
Folder raw      : C:\DA_file\Voicelove\pengangguran-pendidikan-indonesia\data\raw
Folder processed: C:\DA_file\Voicelove\pengangguran-pendidikan-indonesia\data\processed


## 2. Membaca layout tabel seri panjang BPS

Tabel "seri panjang" BPS (2008–2024) menyimpan **dua periode survei dalam satu blok tahun**:

```
baris 1 : Pendidikan Tertinggi ... | 2008 Februari | ... | 2008 Agustus | ...
baris 2 : (kosong)                 | Angkatan Kerja (AK) | ... | Angkatan Kerja (AK) | ...
baris 3 : (kosong)                 | Bekerja | Pengangguran | Jumlah AK | %Bekerja/AK | ...
baris 4 : Tidak/belum pernah ...   | 5,670,322 | 79,764 | 5,750,086 | 98.61 | ...
...
```

Artinya kolom 1–10 = **Februari**, kolom 11–20 = **Agustus**. Karena itu kita perlu
`find_blocks()` untuk mencari baris berisi pola `YYYY Februari`, lalu membaca 18 baris di bawahnya.

In [2]:
df_raw = read_raw("bps_pendidikan_jenis_kegiatan_2008_2024.csv")
print("bentuk file mentah:", df_raw.shape)

blok = find_blocks(df_raw)
print("jumlah blok tahun:", len(blok))
print("3 blok pertama   :", blok[:3])

# isi satu blok apa adanya (kolom 0-4 dan 11-14 saja agar muat di layar)
tahun, mulai, selesai, _ = blok[-1]
print(f"\nContoh blok terakhir (data mulai baris {mulai}):")
df_raw.iloc[mulai - 3 : mulai + 3, [0, 1, 2, 3, 11, 12, 13]].to_string()

bentuk file mentah: (247, 21)
jumlah blok tahun: 17
3 blok pertama   : [(2008, 5, 16, 'Februari'), (2009, 19, 30, 'Februari'), (2010, 33, 44, 'Februari')]

Contoh blok terakhir (data mulai baris 229):


'                                       0                    1             2              3                    11            12             13\n226  Pendidikan Tertinggi yang Ditamatkan        2024 Februari           NaN            NaN         2024 Agustus           NaN            NaN\n227                                   NaN  Angkatan Kerja (AK)           NaN            NaN  Angkatan Kerja (AK)           NaN            NaN\n228                                   NaN              Bekerja  Pengangguran      Jumlah AK              Bekerja  Pengangguran      Jumlah AK\n229            Tidak/belum pernah sekolah         1,437,402.00     13,598.00   1,451,000.00         1,385,633.00     12,651.00   1,398,284.00\n230                  Tidak/belum tamat SD        16,231,950.00    394,364.00  16,626,314.00        15,927,881.00    361,769.00  16,289,650.00\n231                                    SD        34,283,869.00    857,486.00  35,141,355.00        34,471,623.00    853,738.00  35,325,361.00

Blok terakhir = **2024 Agustus**. Perhatikan baris pertamanya berisi label tahun, baris kedua
header `Angkatan Kerja (AK)`, baris ketiga sub-header, baru setelah itu data. Karena itu
`parse_kegiatan()` mulai membaca di `baris_tahun + 3`.

In [3]:
contoh = parse_kegiatan("bps_pendidikan_jenis_kegiatan_2008_2024.csv", "pendidikan_bps")
print("baris hasil parsing:", len(contoh))
contoh[["pendidikan_bps", "periode", "bekerja", "pengangguran", "angkatan_kerja", "tpak", "tpt"]].head(8)

baris hasil parsing: 339


,pendidikan_bps,periode,bekerja,pengangguran,angkatan_kerja,tpak,tpt
0,Tidak/belum pernah sekolah,2008-02,5670322.0,79764.0,5750086.0,57.97,1.39
1,Tidak/belum pernah sekolah,2008-08,5468130.0,103206.0,5571336.0,56.65,1.85
2,Tidak/belum tamat SD,2008-02,12982233.0,448431.0,13430664.0,69.88,3.34
3,Tidak/belum tamat SD,2008-08,13108831.0,443832.0,13552663.0,69.36,3.27
4,SD,2008-02,36963023.0,2216748.0,39179771.0,69.32,5.66
5,SD,2008-08,36756953.0,2099968.0,38856921.0,69.36,5.40
6,SLTP,2008-02,19396319.0,2166619.0,21562938.0,58.68,10.05
7,SLTP,2008-08,19039193.0,1973986.0,21013179.0,56.93,9.39


## 3. Keputusan standardisasi

### 3.1 Pendidikan — dua tingkat label

| Label BPS (2008–2024) | `pendidikan` | `kelompok_pendidikan` (6 level) |
|---|---|---|
| Tidak/belum pernah sekolah | Tidak/belum pernah sekolah | SD ke bawah |
| Tidak/belum tamat SD | Tidak/belum tamat SD | SD ke bawah |
| SD | SD | SD ke bawah |
| SLTP | SMP | SMP |
| SLTA Umum/SMU | SMA | SMA |
| SLTA Kejuruan/SMK | SMK | SMK |
| Akademi/Diploma | Diploma | Diploma |
| Universitas | Sarjana | Sarjana |
| Total | Indonesia | Indonesia |

Kenapa dua kolom? Karena tabel 2025 hanya punya 6 kategori (SD ke bawah sudah digabung).
`pendidikan` mempertahankan detail asli BPS; `kelompok_pendidikan` dipakai kalau kita
membandingkan lintas tahun, termasuk 2025/2026.

### 3.2 Hal lain yang diputuskan

- **`Tak Terjawab` dibuang** dari tabel analisis: angkatan kerjanya 0 di semua periode, sehingga TPT tidak terdefinisi (bukan nol). Jumlah baris yang dibuang dicatat di log validasi.
- **Provinsi** diseragamkan ke 38 provinsi (penamaan 2024). Provinsi hasil pemekaran Papua (92, 95, 96, 97) hanya muncul mulai 2023; Kalimantan Utara kosong 2008–2014 karena belum disurvei terpisah.
- **Blok 2024 ganda** pada tabel status pekerjaan: BPS mengulang blok 2024 dengan 6 kategori (versi harmonisasi). Blok 8 kategori dipakai, blok duplikat dilewati supaya angka tidak terhitung dua kali. Kejadian ini tercatat di `_validasi.csv`.
- **Persen TPT dihitung ulang** dari `pengangguran / angkatan_kerja`, bukan disalin dari tampilan web, agar konsisten dengan angka absolutnya. Kolom `sumber_tpt` menandai hal ini.

In [4]:
pend = build_nasional_pendidikan()
print("bentuk:", pend.shape)
pend.head(3)

bentuk: (306, 16)


,periode,tahun,periode_survei,pendidikan_bps,pendidikan,kelompok_pendidikan,bekerja,pengangguran,angkatan_kerja,bukan_angkatan_kerja,penduduk_usia_kerja,tpak,tpt,sumber_tpt,catatan,sumber_file
0,2008-02,2008,Februari,Tidak/belum pernah sekolah,Tidak/belum pernah sekolah,SD ke bawah,5670322.0,79764.0,5750086.0,4168835.0,9918921.0,57.97,1.39,dihitung (pengangguran/AK),,bps_pendidikan_jenis_kegiatan_2008_2024.csv
1,2008-08,2008,Agustus,Tidak/belum pernah sekolah,Tidak/belum pernah sekolah,SD ke bawah,5468130.0,103206.0,5571336.0,4263060.0,9834396.0,56.65,1.85,dihitung (pengangguran/AK),,bps_pendidikan_jenis_kegiatan_2008_2024.csv
2,2008-02,2008,Februari,Tidak/belum tamat SD,Tidak/belum tamat SD,SD ke bawah,12982233.0,448431.0,13430664.0,5789697.0,19220361.0,69.88,3.34,dihitung (pengangguran/AK),,bps_pendidikan_jenis_kegiatan_2008_2024.csv


In [5]:
# Cek silang dengan angka resmi BPS yang sudah diverifikasi di Tahap 1
for per in ["2024-02", "2024-08"]:
    g = pend[pend["periode"] == per].set_index("pendidikan")["tpt"]
    print(f"{per}  Indonesia={g['Indonesia']:.2f}  SMK={g['SMK']:.2f}  SMA={g['SMA']:.2f}  "
          f"Diploma={g['Diploma']:.2f}  Sarjana={g['Sarjana']:.2f}")

2024-02  Indonesia=4.82  SMK=8.62  SMA=6.73  Diploma=4.87  Sarjana=5.63
2024-08  Indonesia=4.91  SMK=9.01  SMA=7.05  Diploma=4.83  Sarjana=5.25


### Hasil cek silang

| Periode | Indonesia | SMK | SMA | Diploma | Sarjana |
|---|---|---|---|---|---|
| Feb 2024 | 4,82 | 8,62 | 6,73 | 4,87 | 5,63 |
| Agu 2024 | 4,91 | 9,01 | 7,05 | 4,83 | 5,25 |

Angka ini **persis sama** dengan publikasi BPS (Berita Resmi Statistik, TPT menurut pendidikan).
Artinya parser membaca kolom yang benar.

## 4. Validasi otomatis

Empat aturan yang diuji untuk setiap tabel:

1. tidak ada duplikat pada kunci utama (`periode` + dimensi),
2. nilai persen berada di 0–100,
3. `pengangguran <= angkatan_kerja`,
4. jumlah rincian = baris `Indonesia` (uji keseimbangan neraca).

In [6]:
log = []
log += validate(pend, "tpt_nasional_pendidikan", "tpt", ["periode", "pendidikan"])
log += cek_konsistensi(pend, "tpt_nasional_pendidikan")
log += cek_jumlah_bagian(pend, "tpt_nasional_pendidikan", "pendidikan", "Indonesia")
pd.DataFrame(log)

,tabel,cek,nilai,status,catatan
0,tpt_nasional_pendidikan,jumlah baris,306.0,info,
1,tpt_nasional_pendidikan,"duplikat pada ['periode', 'pendidikan']",0.0,OK,
2,tpt_nasional_pendidikan,tpt di luar 0-100,0.0,OK,"min=0.77, max=17.26"
3,tpt_nasional_pendidikan,tpt kosong (NaN),0.0,info,"kosong dibiarkan, tidak diimputasi"
4,tpt_nasional_pendidikan,pengangguran > angkatan kerja,0.0,OK,
5,tpt_nasional_pendidikan,jumlah rincian vs baris Indonesia (orang),0.0,OK,tidak ada selisih


Semua **OK**. Dua hal yang perlu dibaca sebagai catatan, bukan kegagalan:

- `tpt_provinsi` punya 14 sel kosong = Kalimantan Utara 2008–2014 (7 tahun x 2 periode). Provinsi ini baru dimekarkan 2012 dan baru disurvei terpisah pada Sakernas 2015.
- Baris `Tak Terjawab` (33 baris) dihapus dari tabel pendidikan karena AK = 0.

Nol imputasi dilakukan untuk keduanya. Uji silang tambahan juga lolos: total pekerja dari tabel status pekerjaan **sama persis** dengan `bekerja` di tabel pendidikan untuk semua 34 periode, dan jumlah pengangguran 38 provinsi sama dengan total nasional.

In [7]:
prov = build_provinsi()
print("provinsi unik:", prov["provinsi"].nunique(), "| termasuk 'Indonesia':", "Indonesia" in prov["provinsi"].unique())

terbaru = prov[(prov["periode"] == "2024-08") & (prov["provinsi"] != "Indonesia")]
print("\n5 TPT tertinggi, Agustus 2024:")
print(terbaru.nlargest(5, "tpt")[["provinsi", "tpt", "pengangguran"]].to_string(index=False))
print("\n5 TPT terendah, Agustus 2024:")
print(terbaru.nsmallest(5, "tpt")[["provinsi", "tpt", "pengangguran"]].to_string(index=False))

provinsi unik: 39 | termasuk 'Indonesia': True

5 TPT tertinggi, Agustus 2024:
        provinsi  tpt  pengangguran
      Jawa Barat 6.75     1768226.0
          Banten 6.68      414754.0
Papua Barat Daya 6.48       20627.0
           Papua 6.48       31773.0
       Kep. Riau 6.39       71572.0

5 TPT terendah, Agustus 2024:
           provinsi  tpt  pengangguran
   Papua Pegunungan 1.32       13485.0
               Bali 1.79       48676.0
     Sulawesi Barat 2.68       20876.0
Nusa Tenggara Barat 2.73       87011.0
       Papua Tengah 2.75       22047.0


In [8]:
# Provinsi hasil pemekaran Papua: kapan mulai muncul?
papua = prov[prov["kode_provinsi"].isin(["91", "92", "94", "95", "96", "97"])]
print(papua.groupby(["kode_provinsi", "provinsi"])["periode"].agg(["min", "max", "count"]).to_string())

                                    min      max  count
kode_provinsi provinsi                                 
91            Papua Barat       2008-02  2024-08     34
92            Papua Barat Daya  2024-02  2024-08      2
94            Papua             2008-02  2024-08     34
95            Papua Selatan     2024-02  2024-08      2
96            Papua Tengah      2024-02  2024-08      2
97            Papua Pegunungan  2024-02  2024-08      2


Tabel di atas penting untuk *chart* peta/wilayah nanti: provinsi 92, 95, 96, 97 **tidak punya data
sebelum 2023**. Kalau dibuat tren 2015–2024, keempatnya harus dimulai dari 2023 dan diberi catatan.

In [9]:
umur = build_umur()
print("kelompok umur:", umur["kelompok_umur"].unique())
print(umur[umur["periode"] == "2024-08"][["kelompok_umur", "pengangguran", "angkatan_kerja", "tpt"]].to_string(index=False))

kelompok umur: <ArrowStringArray>
['15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', '60 +', 'Total']
Length: 11, dtype: str
kelompok_umur  pengangguran  angkatan_kerja   tpt
      15 - 19     1431960.0       6409538.0 22.34
      20 - 24     2497979.0      16279127.0 15.34
      25 - 29     1258842.0      17622224.0  7.14
      30 - 34      584544.0      17365426.0  3.37
      35 - 39      392036.0      17430430.0  2.25
      40 - 44      321531.0      16922644.0  1.90
      45 - 49      275437.0      16142538.0  1.71
      50 - 54      247373.0      14125656.0  1.75
      55 - 59      182491.0      11460666.0  1.59
         60 +      273406.0      18349354.0  1.49
        Total     7465599.0     152107603.0  4.91


Pola yang sudah terlihat: TPT **15–19 tahun** belasan kali lipat TPT 45+.
Ini akan jadi salah satu temuan utama di Tahap 3, tapi belum kita simpulkan sekarang.

In [10]:
status_long, formal_informal = build_status_pekerjaan()

print("baris status x pendidikan:", len(status_long))
print("\nPorsi pekerja informal menurut pendidikan, Agustus 2024:")
t = formal_informal[formal_informal["periode"] == "2024-08"].sort_values("porsi_informal", ascending=False)
print(t[["pendidikan", "Formal", "Informal", "total_bekerja", "porsi_informal"]].to_string(index=False))

baris status x pendidikan: 2736

Porsi pekerja informal menurut pendidikan, Agustus 2024:
                pendidikan     Formal   Informal  total_bekerja  porsi_informal
Tidak/belum pernah sekolah    80367.0  1305266.0      1385633.0           94.20
      Tidak/belum tamat SD  2516027.0 13411854.0     15927881.0           84.20
                        SD  8284352.0 26187271.0     34471623.0           75.97
                       SMP  8388771.0 17088994.0     25477765.0           67.07
                 Indonesia 60815798.0 83826206.0    144642004.0           57.95
                       SMA 15070027.0 15164482.0     30234509.0           50.16
                       SMK 11452788.0  7139982.0     18592770.0           38.40
                   Diploma  2473190.0   885906.0      3359096.0           26.37
                   Sarjana 12550276.0  2642451.0     15192727.0           17.39


Definisi formal/informal mengikuti BPS:

- **Formal** = (3) berusaha dibantu buruh tetap/dibayar + (4) buruh/karyawan/pegawai
- **Informal** = (1) berusaha sendiri, (2) berusaha dibantu buruh tidak tetap, (5) pekerja bebas pertanian, (6) pekerja bebas non-pertanian, (7) pekerja keluarga/tak dibayar

Angka `Indonesia` di tabel ini bisa dipakai sebagai pembanding: porsi informal nasional
sekitar 59% (2023) — sangat tinggi, dan itulah konteks kenapa "dapat kerja" belum tentu "kerja layak".

In [11]:
# Catatan keputusan pembersihan yang terekam otomatis selama proses
pd.DataFrame(CATATAN)

,tabel,cek,nilai,status,catatan
0,tpt_nasional_pendidikan,"baris 'Tak terjawab' dibuang (AK=0, TPT tidak ...",33,info,kategori tetap ada di raw; hanya dihapus dari ...
1,bps_status_pekerjaan_pendidikan_2008_2024.csv,blok 2024 ganda dilewati,1,info,BPS mengulang blok 2024 dengan 6 kategori; dip...


In [12]:
snap = build_snapshot_2025()
print("snapshot 2025 — dimensi:", snap["dimensi"].unique(), "| baris:", len(snap))
print(snap[["dimensi", "kategori", "nilai"]].to_string(index=False))

snapshot 2025 — dimensi: <ArrowStringArray>
['pendidikan', 'jenis_kelamin', 'daerah', 'kelompok_umur']
Length: 4, dtype: str | baris: 22
      dimensi                                          kategori  nilai
   pendidikan Tidak/Belum Pernah Sekolah/Belum Tamat & Tamat SD   2.30
   pendidikan                                               SMP   3.80
   pendidikan                                          SMA umum   6.88
   pendidikan                                      SMA Kejuruan   8.63
   pendidikan                                  Diploma I/II/III   4.31
   pendidikan                                       Universitas   5.39
jenis_kelamin                                         Laki-Laki   4.85
jenis_kelamin                                         Perempuan   4.84
       daerah                                         Perkotaan   5.75
       daerah                                         Perdesaan   3.47
       daerah                             Perkotaan + Perdesaan   4.85
kelompok_um

In [13]:
informal23 = build_informal_2023()
setengah25 = build_setengah_pengangguran_2025()

print("Porsi informal menurut pendidikan, 2023:")
print(informal23[["pendidikan", "nilai"]].to_string(index=False))
print("\nSetengah pengangguran menurut pendidikan, 2025:")
print(setengah25[["pendidikan", "nilai"]].to_string(index=False))

Porsi informal menurut pendidikan, 2023:
    pendidikan  nilai
 Tidak sekolah  93.00
Tidak tamat SD  85.04
            SD  77.35
           SMP  68.09
           SMA  50.34
           SMK  38.97
       Diploma  27.35
       Sarjana  17.30
     Indonesia  59.11

Setengah pengangguran menurut pendidikan, 2025:
 pendidikan  nilai
SD ke bawah   8.81
        SMP   8.58
        SMA   8.34
        SMK   6.81
    Diploma   4.58
    Sarjana   5.29


Perhatikan arah yang berlawanan:

- **TPT** tinggi di SMK/SMA, rendah di SD ke bawah.
- **Porsi informal** justru tinggi di SD ke bawah (93%) dan rendah di Sarjana (17,3%).

Artinya lulusan rendah "tidak menganggur" karena menerima kerja apa saja — termasuk kerja informal
yang jam kerjanya sering kurang dari 35 jam/minggu (setengah pengangguran 8,8%).
Ini *insight* inti yang akan divisualisasikan di Tahap 3.

In [14]:
tri = pd.concat(
    [
        parse_triwulanan("bps_tpt_provinsi_triwulanan_2025.csv", 2025),
        parse_triwulanan("bps_tpt_provinsi_triwulanan_2026.csv", 2026),
    ],
    ignore_index=True,
)
print("periode triwulanan tersedia:", sorted(tri["periode"].unique()))
print("catatan: Mei 2025 kosong di sumber (Sakernas Mei 2025 tidak diterbitkan)")

nas = tri[tri["provinsi"] == "Indonesia"]
print("\nTPT nasional triwulanan:")
print(nas[["periode", "tpt"]].to_string(index=False))

periode triwulanan tersedia: ['2025-02', '2025-08', '2025-11', '2026-02', '2026-05', '2026-08']
catatan: Mei 2025 kosong di sumber (Sakernas Mei 2025 tidak diterbitkan)

TPT nasional triwulanan:
periode  tpt
2025-02 4.76
2025-08 4.85
2025-11 4.74
2026-02 4.68
2026-05 4.65
2026-08 4.75


In [15]:
tpak = parse_tpak_provinsi("bps_tpak_provinsi_2026.csv", 2026)
print("TPAK 2026 — periode:", tpak["periode"].unique(), "| provinsi:", tpak["provinsi"].nunique())
print(tpak.nlargest(5, "tpak")[["provinsi", "tpak"]].to_string(index=False))

TPAK 2026 — periode: <ArrowStringArray>
['2026-02']
Length: 1, dtype: str | provinsi: 39


           provinsi  tpak
   Papua Pegunungan 85.87
       Papua Tengah 81.74
Nusa Tenggara Timur 76.94
Nusa Tenggara Barat 76.18
         Jawa Timur 74.78


## 5. Ringkasan output

| File di `data/processed/` | Isi | Baris |
|---|---|---|
| `tpt_nasional_pendidikan.csv` | TPT nasional per pendidikan, Feb & Agu 2008–2024 | 306 |
| `tpt_provinsi.csv` | TPT 38 provinsi + Indonesia, 2008–2024 | 1.198 |
| `tpt_nasional_umur.csv` | TPT nasional per kelompok umur | 374 |
| `status_pekerjaan_pendidikan.csv` | Jumlah pekerja per status x pendidikan | 2.849 |
| `formal_informal_pendidikan.csv` | Porsi formal/informal per pendidikan | 306 |
| `snapshot_2025.csv` | TPT 2025: pendidikan, gender, kota/desa, umur | 22 |
| `informal_pendidikan_2023.csv` | Porsi informal per pendidikan (2023) | 9 |
| `setengah_pengangguran_pendidikan_2025.csv` | Setengah pengangguran per pendidikan (2025) | 6 |
| `tpt_provinsi_triwulanan.csv` | TPT provinsi triwulanan 2025–2026 | 234 |
| `tpak_provinsi.csv` | TPAK provinsi 2026 | 39 |
| `_validasi.csv` | Log validasi & keputusan pembersihan | — |

Kesimpulan Tahap 2: **semua tabel lolos validasi**, angka kunci cocok dengan publikasi BPS,
dan tidak ada nilai yang diimputasi.

In [16]:
for f in sorted(PROC.glob("*.csv")):
    print(f"{f.name:48s} {len(pd.read_csv(f)):5d} baris  {f.stat().st_size/1024:7.1f} KB")

_validasi.csv                                       45 baris      3.4 KB
formal_informal_pendidikan.csv                     306 baris     41.5 KB
informal_pendidikan_2023.csv                         9 baris      0.8 KB
setengah_pengangguran_pendidikan_2025.csv            6 baris      0.6 KB
snapshot_2025.csv                                   22 baris      2.8 KB
status_pekerjaan_pendidikan.csv                   2736 baris    342.0 KB
tpak_provinsi.csv                                   39 baris      2.3 KB


tpt_nasional_pendidikan.csv                        306 baris     56.8 KB
tpt_nasional_umur.csv                              374 baris     60.1 KB
tpt_provinsi.csv                                  1198 baris    202.2 KB
tpt_provinsi_triwulanan.csv                        234 baris     15.4 KB
